# 02 — Dataset assembly

Turns 9 sessions of raw recording into one table a model can train on, plus an honest
evaluation design. **Trains nothing.**

Exit criteria: a merged table in memory, 9 leave-one-session-out folds, a written
`split_manifest.json`, and assertions proving nothing leaks.

Plan: `docs/logs/20260810_130412_handoff_notebook02.md`

## Section 0 — Setup

Imports, repo-root bootstrap, and the run-level constants.

The notebook's kernel starts in `notebooks/`, so the repo root is not on `sys.path` and
`core.features` will not import until it is put there. The `main.py` assert catches the
case where the kernel's working directory is not what we assume.

| Constant | Meaning |
|---|---|
| `RANDOM_SEED` | Touches only the secondary trial-level split. The 9 session folds are deterministic — the headline number does not depend on a seed |
| `REBUILD_FEATURES` | Always regenerate the feature cache. `False` only when a rebuild is slow and nothing has changed |
| `CALIBRATION_MODE` | `per_wear` / `none` / `stale`. Recorded into `features_meta.json`; must never influence fold membership |

In [ ]:
import pandas as pd 
import numpy as np
from pathlib import Path
import json
import sys

CURRENT_FOLDER = Path.cwd()
ROOT_FOLDER = CURRENT_FOLDER.parent
assert (ROOT_FOLDER / 'main.py').exists(), f'not the repo root: {ROOT_FOLDER}'

root_str = str(ROOT_FOLDER)
if root_str not in sys.path:
    sys.path.insert(0, root_str)

from core.features import to_grid, max_disp, strain,  velocity              # executed after insert(0, ...) since kernel does not know root folder

PARTICIPANT = ROOT_FOLDER / 'recordings' / 'participant001_20260805_092548'

RANDOM_SEED = 42
REBUILD_FEATURES = True
CALIBRATION_MODE = 'per_wear'

## Section 1 — Session discovery and loading

Find the usable sessions, then load one at a time into `(df, layout)`.

**Two traps on disk.** The participant folder holds an `aborted/` directory alongside the
10 sessions, containing 5 abandoned attempts whose names (`session1_…`, `session3_…`,
`session7_…`) collide with real ones — `glob('session*')` excludes it by name, so it must
not become `iterdir()` or `rglob()`. And sorting folder names lexicographically puts
`session10` *second*; the sort key is the parsed integer instead.

**Session 4 is unlabelled, not broken.** Its BIOPAC force export was corrupted, so it has
`markers.csv` and `grid_layout.json` but no `training_table.csv`. Discovery filters on that
file's existence, which is why there are **9 sessions, not 10**.

Discovery and loading are kept apart deliberately: `usable_sessions()` decides what is
usable, so `load_session()` can stay strict and has no failure branch to tolerate.

**Verified state after this section** (outputs are stripped on commit, so the numbers live here):

| | |
|---|---|
| Usable sessions | 9 — numbered **1, 2, 3, 5, 6, 7, 8, 9, 10** |
| Columns per session | 405 — 9 meta + 4 channels × 99 markers |
| Layout | `(99, 2)`, identical across all 9 sessions |
| Rows per session | 2,751 – 3,667 (they differ; never reuse a mask across sessions) |
| **Total rows** | **28,261** — pre-filter. Section 3 drops 1 |

In [ ]:
"""Session discovery helpers.

session_number  -- pulls the integer out of a folder name like
                   'session10_20260805_111615'. Used as the sort key here and as
                   session_id later, so both agree by construction.

usable_sessions -- the session folders that actually carry a training_table.csv.
                   glob('session*') is what keeps the aborted/ folder out: it holds
                   5 abandoned attempts whose names collide with real sessions.
                   Do NOT switch this to iterdir() or rglob().
"""

def session_number(session_dir):
    removed_session = session_dir.name.removeprefix('session')
    return int(removed_session.split('_')[0])


def usable_sessions(participant_dir):
    return sorted(
        [
            folder for folder in participant_dir.glob('session*')
            if (folder / 'training_table.csv').exists()
        ],
        key=session_number
    )

In [ ]:
SESSIONS = usable_sessions(PARTICIPANT)

# Every session* folder on disk, usable or not -- so the two sets can be compared.
all_found = {session_number(s) for s in PARTICIPANT.glob('session*')}
usable_found = {session_number(s) for s in SESSIONS}

assert len(SESSIONS) == 9, [s.name for s in SESSIONS]

# Session 4's BIOPAC export was corrupted, so it has no training_table.csv.
# It must be the ONLY one excluded -- a bare len == 9 would also pass if session 4
# were fixed while some other session lost its table.
assert all_found - usable_found == {4}, (
    f'expected only session 4 to be unlabelled, '
    f'but excluded {sorted(all_found - usable_found)}'
)

print(f'{len(SESSIONS)} usable sessions:', sorted(usable_found))

In [ ]:
def load_session(session_dir):
    """Load one session's training table and its marker grid layout.

    Returns (df, layout). `layout` is an (n_markers, 2) int array indexed by
    marker_id, so layout[i] is marker i's (row, col) on the sensor.

    Strict on purpose: no existence check, no None return. Only folders that came
    out of usable_sessions() should ever reach here, so a missing file is a bug in
    the caller, not a data condition to tolerate.
    """
    df = pd.read_csv(session_dir / 'training_table.csv')

    with (session_dir / 'grid_layout.json').open() as f:
        meta = json.load(f)

    # A fallback_sort layout has no geometric meaning and is written as null.
    if meta['source'] != 'geometry':
        raise ValueError(
            f"{session_dir.name}: grid layout source is {meta['source']!r}, "
            f"not 'geometry' (reason: {meta['reason']})"
        )

    n_rows = meta['grid_shape']['n_rows']
    n_cols = meta['grid_shape']['n_cols']
    n_markers = n_rows * n_cols

    # JSON object keys are always strings, and their order is arbitrary --
    # allocate first, then assign by marker_id.
    layout = np.full((n_markers, 2), -1, dtype=int)
    for marker_id, pos in meta['layout'].items():
        layout[int(marker_id)] = [pos['row'], pos['col']]

    if (layout < 0).any():
        missing = np.flatnonzero(layout[:, 0] < 0).tolist()
        raise ValueError(f'{session_dir.name}: no position for marker ids {missing}')

    return df, layout

In [ ]:
N_MARKERS = 99
N_COLUMNS = 405   # 9 meta columns + 4 channels x 99 markers

reference_layout = None
row_counts = {}

for session_dir in SESSIONS:
    n = session_number(session_dir)
    df, layout = load_session(session_dir)

    # Column count is the contract every later section relies on. Row counts differ
    # per session (2,751-3,667), so never assert those.
    assert df.shape[1] == N_COLUMNS, f'session {n}: {df.shape[1]} columns, want {N_COLUMNS}'
    assert layout.shape == (N_MARKERS, 2), f'session {n}: layout {layout.shape}'

    # The grid scatter must be the same physical arrangement in every session, or
    # strain would be computed across different neighbours in different mounts.
    if reference_layout is None:
        reference_layout = layout
    assert np.array_equal(layout, reference_layout), f'session {n}: layout differs'

    row_counts[n] = len(df)

# Captured here, while the sessions are still separate -- the final assertions section
# checks the merged row count against it.
TOTAL_ROWS = sum(row_counts.values())

print(f'{len(SESSIONS)} sessions, {N_COLUMNS} columns each, layout {reference_layout.shape}')
print(f'rows per session: {row_counts}')
print(f'total rows: {TOTAL_ROWS:,}  (pre-filter; the trial filter drops 1)')

## Section 2 — Time to cue

`target_pct` labels a whole trial, but roughly half of every trial is the **pre-cue wait**,
where the participant sits at ~0.7–1.1 % MVC no matter whether the trial is labelled 5 % or
80 %. The real rest/grip boundary is `grip_start` — the operator's "go" press, logged per
trial in `events.json`.

`cue_offset()` turns that into seconds relative to the cue: **negative before it, positive
after**. Section 4 slices `phase` and `braced` out of this, and Section 3's trial filter keys
on its NaNs.

**The NaN is load-bearing.** A trial with no `grip_start` event gets `NaN` for every one of
its frames, and `NaN >= 0` is `False` — so an uncued trial can never be mislabelled
`post_cue`. A sentinel like `-999` would quietly pass as "pre-cue" forever. Keep the
`cue.get(t, np.nan)`; do not turn it into `cue[t]`.

**Verified against session 2** (the only session with a missing cue):

| | |
|---|---|
| `grip_start` events | 29, for 30 trials |
| Missing cue | `trial_id` **13** — one frame long, dropped in Section 3 |
| `to_cue` range | −8.09 s to +8.44 s |
| NaN frames | 1 |

Two things confirmed on disk rather than assumed. Event `ts` values sit inside the CSV's own
`timestamp_ms` range, so both are the same clock in the same units — the subtraction needs no
offset correction. And `events.json` holds a `video_log` entry alongside the cues, which is
why the `type == 'grip_start'` filter is not decoration.

`trial_id` runs 0–29 contiguously with no sentinel for inter-trial frames, so there is no
"outside a trial" case to handle here.

In [ ]:
def cue_offset(session_dir, df):
    """Seconds from each frame to its trial's grip_start cue. Negative = before it.

    grip_start is the operator's "go" press, logged per trial in events.json.
    It is the real rest/grip boundary: target_pct labels the whole trial, and
    roughly half of every trial is the pre-cue wait, where force sits at
    ~0.7-1.1% MVC no matter what the trial is labelled.
    """
    with (session_dir / 'events.json').open() as f:
        events = json.load(f)['events']

    # events.json also carries video_log entries -- filtering by type is not optional.
    cue = {e['trial_id']: e['ts'] for e in events if e['type'] == 'grip_start'}

    # NaN for any trial with no cue -- honest, and it fails loudly downstream.
    cue_ts = np.array([cue.get(t, np.nan) for t in df['trial_id'].to_numpy()])

    return (df['timestamp_ms'].to_numpy() - cue_ts) / 1000.0

In [ ]:
# Smoke test on session 2 -- the one session with a missing cue, so it exercises
# both the normal path and the NaN path. The sweep across all 9 sessions belongs to
# Section 3, where the trial filter lives; doing it here too would be two places to
# update when the filter changes.

s2_dir = next(s for s in SESSIONS if session_number(s) == 2)
s2_df, _ = load_session(s2_dir)

to_cue = cue_offset(s2_dir, s2_df)

assert len(to_cue) == len(s2_df)

# Both signs must be present, or the cue is outside the recording and the clocks
# do not line up -- the failure this would otherwise hide.
assert (to_cue < 0).any() and (to_cue > 0).any(), 'cue lies outside the session'

uncued = s2_df.loc[np.isnan(to_cue), 'trial_id'].unique()
assert uncued.tolist() == [13], f'expected only trial 13 uncued, got {uncued.tolist()}'

print(f'session 2: {np.isnan(to_cue).sum()} uncued frame(s), trial(s) {uncued.tolist()}')
print(f'to_cue range: {np.nanmin(to_cue):.2f} s to {np.nanmax(to_cue):+.2f} s')

## Section 3 — Trial filter

**The only deletion in this notebook.** A trial with no `grip_start` was never cued, so it is
not a hard trial — it is a non-trial. Everything else gets flagged, not dropped.

`to_cue` is attached as a column here so the mask and the rows move together through the drop
and the later concat. Section 4 reads `phase` and `braced` off it.

| | |
|---|---|
| Trials | 270 → **269** |
| Frames | 28,261 → **28,260** |
| Dropped | session 2, trial 13 — an 80 % trial, 1 frame long |

Session 2 therefore holds 5 × 80 % trials; every other session keeps its full 6.

Do **not** filter on a force threshold (4.57 % MVC would also catch legitimate 5 % trials) and
do **not** rely on `voided` — it is `False`, because the app saw nothing wrong.

In [ ]:
def drop_uncued(session_dir, df):
    """Attach to_cue and drop the frames of any trial that was never cued.

    assign() returns a new frame, so the caller's df is left alone. reset_index
    closes the hole the drop leaves -- later sections assign numpy arrays back as
    columns, and pandas aligns those on the index.
    """
    df = df.assign(to_cue=cue_offset(session_dir, df))

    return df[df['to_cue'].notna()].reset_index(drop=True)

In [ ]:
TOTAL_TRIALS = 269       # 270 collected, 1 uncued
TOTAL_ROWS_FILTERED = TOTAL_ROWS - 1

dropped = []             # (session_id, trial_id) of everything the filter removed
trials_after = {}
rows_after = {}

for session_dir in SESSIONS:
    n = session_number(session_dir)
    df, _ = load_session(session_dir)

    kept = drop_uncued(session_dir, df)

    gone = set(df['trial_id']) - set(kept['trial_id'])
    dropped += [(n, int(t)) for t in sorted(gone)]

    trials_after[n] = kept['trial_id'].nunique()
    rows_after[n] = len(kept)

# Naming the trial, not just counting it. A count alone would still pass if the
# filter started dropping a different trial for a different reason.
assert dropped == [(2, 13)], f'expected only session 2 trial 13, got {dropped}'

assert sum(trials_after.values()) == TOTAL_TRIALS
assert sum(rows_after.values()) == TOTAL_ROWS_FILTERED

print(f'dropped {dropped} -> {TOTAL_TRIALS} trials, {TOTAL_ROWS_FILTERED:,} frames')
print(f'trials per session: {trials_after}')

## Section 4 — Flag columns

Three flags, **no rows removed**. Each one is a reporting axis later: "8 % error during
squeezes, 22 % during bracing" is a stronger and more defensible result than one averaged
number, and flagging is reversible where trimming is a one-way door.

| Flag | Rule | Why it exists |
|---|---|---|
| `phase` | `to_cue >= 0` → `post_cue` | ~half of every trial is the pre-cue wait, sitting at ~0.7–1.1 % MVC *whatever* the target says |
| `braced` | final 1 s before the cue | The "3, 2, 1" countdown moves markers +3.2 px while force moves only 0.57 → 1.83 % MVC — the sensor sees tension the dynamometer does not |
| `carryover` | grip trial still above 10 % MVC over its final 0.5 s | Operator advanced before the participant released. 8 of 215 grip trials; their release transient is truncated |

`carryover` is **recomputed from the rule**, then asserted against the 8 trials named in the
labelling log §2 — so the notebook derives the list and confirms the doc, rather than
inheriting a magic set from a file with no version history.

`braced` is a subset of `pre_cue`; the two are meant to overlap.

**Verified state after this section:**

| | |
|---|---|
| `pre_cue` frames | **13,705** — independently reproduces the labelling log §8 figure |
| `post_cue` frames | 14,555 |
| `braced` frames | 4,365 |
| `carryover` | 8 trials, 959 frames — the rule reproduces §2's list exactly, no extras, no misses |

Roughly **48 % of all frames are pre-cue**, which is the single fact that makes `phase`
necessary rather than decorative.

In [ ]:
BRACED_SEC = 1.0          # window before the cue that counts as anticipatory bracing
CARRYOVER_TAIL_MS = 500   # tail of a grip trial used to test for an incomplete release
CARRYOVER_PCT = 10.0      # mean %MVC over that tail that marks the trial


def add_flags(df):
    """Add phase / braced / carryover. Row count is unchanged.

    phase and braced are per-frame; carryover is a per-trial verdict broadcast
    back to every frame of that trial, which is why it needs the groupby.
    """
    trial_end = df.groupby('trial_id')['timestamp_ms'].transform('max')
    in_tail = df['timestamp_ms'] >= trial_end - CARRYOVER_TAIL_MS

    # Rest trials are excluded because the rule is defined over grip trials. It is
    # also what keeps a rest trial's own tail from ever qualifying.
    tail_mean = df[in_tail & (df['target_pct'] != 0)].groupby('trial_id')['pct_mvc_corr'].mean()
    flagged = tail_mean.index[tail_mean > CARRYOVER_PCT]

    return df.assign(
        phase=np.where(df['to_cue'] >= 0, 'post_cue', 'pre_cue'),
        braced=(df['to_cue'] >= -BRACED_SEC) & (df['to_cue'] < 0),
        carryover=df['trial_id'].isin(flagged),
    )

In [ ]:
# The 8 trials named in docs/logs/20260808_151738_trial_labelling_and_grip_carryover.md §2.
# Kept only as the thing the recomputed rule is checked against -- not used to build the flag.
CARRYOVER_DOCUMENTED = {(1, 0), (1, 2), (1, 5), (1, 13), (2, 10), (5, 12), (5, 13), (8, 5)}

carryover_found = set()
phase_counts = {'pre_cue': 0, 'post_cue': 0}
braced_frames = 0
flagged_rows = 0

for session_dir in SESSIONS:
    n = session_number(session_dir)
    df, _ = load_session(session_dir)
    df = add_flags(drop_uncued(session_dir, df))

    assert len(df) == rows_after[n], f'session {n}: add_flags changed the row count'

    carryover_found |= {(n, int(t)) for t in df.loc[df['carryover'], 'trial_id'].unique()}
    for p, c in df['phase'].value_counts().items():
        phase_counts[p] += int(c)
    braced_frames += int(df['braced'].sum())
    flagged_rows += int(df['carryover'].sum())

# The rule must reproduce the documented list exactly -- a bare count of 8 would pass
# even if the threshold had drifted onto a different set of trials.
assert carryover_found == CARRYOVER_DOCUMENTED, (
    f'extra {sorted(carryover_found - CARRYOVER_DOCUMENTED)}, '
    f'missing {sorted(CARRYOVER_DOCUMENTED - carryover_found)}'
)

assert sum(phase_counts.values()) == TOTAL_ROWS_FILTERED

print(f'carryover: {len(carryover_found)} trials, {flagged_rows:,} frames')
print(f'phase: {phase_counts}')
print(f'braced: {braced_frames:,} frames (subset of pre_cue)')

## Section 5 — Derive channels

Calls `core/features.py` and flattens the grids back to per-marker columns.

The round trip matters: `to_grid` scatters the 99 markers onto their physical `(11, 9)`
arrangement, which is what `strain` needs to know who its neighbour is. Flattening reverses it
with the same `layout`, so the columns come back in `marker_id` order. **Both directions read
`grid_layout.json` — never `row * 9 + col`.**

| Channel | Columns | |
|---|---|---|
| `max_disp` | 1 | Largest displacement anywhere in the frame |
| `strain_x` / `strain_y` | 99 each | Movement relative to the neighbouring marker |
| `vx` / `vy` | 99 each | Per-marker speed, from real elapsed time |

**Velocity NaN is left bare** — 269 frames (0.95 %), at frame 0 and every trial boundary,
where there is no valid earlier frame to difference against. A 0 would be read as "this marker
did not move", which is a lie. No `valid_velocity` column: nothing else here produces NaN
(`to_grid` raises instead), so `.isna()` is already unambiguous. Notebook 03 decides whether to
drop those rows or use a NaN-tolerant model, once the model's constraints are known.

**Verified state after this section:**

| | |
|---|---|
| Columns | 405 → **806** (405 + `to_cue` + 3 flags + 397 derived) |
| Velocity NaN frames | 269 = 1 + trial boundaries, in every session |
| `strain` / `max_disp` NaN | 0 |
| Round trip | `to_grid` → flatten returns `dx` bit-for-bit |

In [ ]:
DERIVED_GRIDS = ('strain_x', 'strain_y', 'vx', 'vy')   # 99 columns each
DERIVED_SCALARS = ('max_disp',)                        # 1 column, whole grid


def add_channels(df, layout):
    """Derive the notebook-01 survivors and flatten them back to per-marker columns.

    layout[:, 0] / layout[:, 1] index the grid exactly as to_grid scattered it, so
    grid[:, rows, cols] undoes the scatter and returns marker_id order.
    """
    grids = {
        name: to_grid(df[[f'{name}_{i}' for i in range(N_MARKERS)]].to_numpy(), layout)
        for name in ('dx', 'dy', 'mag')
    }

    sx, sy = strain(grids['dx'], grids['dy'])
    vx, vy = velocity(
        grids['dx'], grids['dy'],
        df['timestamp_ms'].to_numpy(), df['trial_id'].to_numpy(),
    )

    derived = {'max_disp': max_disp(grids['mag'])}

    for name, grid in zip(DERIVED_GRIDS, (sx, sy, vx, vy)):
        flat = grid[:, layout[:, 0], layout[:, 1]]     # (n, 99), back in marker_id order
        derived.update({f'{name}_{i}': flat[:, i] for i in range(N_MARKERS)})

    # One concat rather than 397 inserts -- repeated column assignment fragments the frame.
    return pd.concat([df, pd.DataFrame(derived, index=df.index)], axis=1)

In [ ]:
N_COLUMNS_DERIVED = (
    N_COLUMNS                                  # 405 from training_table.csv
    + 1                                        # to_cue
    + 3                                        # phase, braced, carryover
    + len(DERIVED_SCALARS)
    + len(DERIVED_GRIDS) * N_MARKERS
)

velocity_nans = {}

for session_dir in SESSIONS:
    n = session_number(session_dir)
    df, layout = load_session(session_dir)
    df = add_channels(add_flags(drop_uncued(session_dir, df)), layout)

    assert df.shape == (rows_after[n], N_COLUMNS_DERIVED), f'session {n}: {df.shape}'

    # Round trip: scatter dx onto the grid, undo it, and the original must come back.
    # This is the check that catches a transposed or mis-ordered layout, which would
    # otherwise scramble which marker each strain value belongs to -- silently.
    dx_raw = df[[f'dx_{i}' for i in range(N_MARKERS)]].to_numpy()
    dx_grid = to_grid(dx_raw, layout)
    assert np.array_equal(dx_grid[:, layout[:, 0], layout[:, 1]], dx_raw), f'session {n}'

    # Velocity is NaN at frame 0 and every trial boundary -- and nowhere else.
    trials = df['trial_id'].to_numpy()
    expected_nan = 1 + int((trials[1:] != trials[:-1]).sum())
    got_nan = int(df['vx_0'].isna().sum())
    assert got_nan == expected_nan, f'session {n}: {got_nan} NaN, expected {expected_nan}'

    # Only velocity may carry NaN. strain copies its edge, max_disp is a plain max.
    for name in ('strain_x', 'strain_y'):
        assert not df[[f'{name}_{i}' for i in range(N_MARKERS)]].isna().any().any(), name
    assert not df['max_disp'].isna().any()

    velocity_nans[n] = got_nan

print(f'{N_COLUMNS} -> {N_COLUMNS_DERIVED} columns')
print(f'velocity NaN frames: {velocity_nans}')
print(f'total {sum(velocity_nans.values())} of {TOTAL_ROWS_FILTERED:,} '
      f'= {100 * sum(velocity_nans.values()) / TOTAL_ROWS_FILTERED:.2f}%')

df.shape

## Section 6 — Calibration (switchable)

The sleeve sits differently every time it is worn. At rest, before anyone squeezes, mounts read
between **0.93 and 5.09 px** of displacement. Pool them uncorrected and the model can learn
*which session it is looking at* and use mount identity as a shortcut.

The fix is to subtract each session's own resting level. **It must be a switch, not hardcoded
arithmetic** — SO4 says design *and evaluate* the protocol, and one error number evaluates the
model, not the protocol. Only the gap between the three modes evaluates the protocol.

| `mode` | Offset applied | What it answers |
|---|---|---|
| `per_wear` *(default)* | The session's own | The protocol as designed |
| `none` | None | What does calibration buy? |
| `stale` | The **previous** session's, wrapping | Must it be redone every wear? |

`stale` donor: s1←s10, s2←s1, s3←s2, s5←s3, s6←s5, s7←s6, s8←s7, s9←s8, s10←s9. Chosen over a
fixed donor because every session gets a different one, so the result cannot be an artifact of
one unusual mount.

**The offset window is trial 0's pre-cue interval minus the braced second** — the start of the
session, before anything happened. Not the first *rest trial*: trial order is randomised, and
that lands at trial 14 in session 7, which is not a step a user could perform at wear time. The
first-rest-trial window did score better (−40 % vs −31 %) but the difference is **not
significant** across 9 sessions (paired t p = 0.150), so the choice rests on protocol realism,
not on the numbers. See `docs/design/methods/so4-calibration-protocol.md` §4.

**What gets calibrated:** `dx`, `dy`, `dA` — then `mag` is **rebuilt**, because it is stored as
`hypot(dx, dy)` and would otherwise still describe uncalibrated displacement. `dA` is included
because its resting spread across mounts (12.7 units) is the largest of any channel; leaving it
raw would reintroduce the shortcut through that channel.

**Order matters:** calibration runs *before* `add_channels`, so `strain`, `vx`/`vy` and
`max_disp` all inherit the correction from their source instead of each needing their own.

⚠ **Fold membership must never depend on the mode.** Folds come from session and trial identity
only. If a mode change moves the folds, the contrast compares two different experiments.

### Verified state after this section

Resting displacement scored on **post-cue rest frames**, which are disjoint from the calibration
window — scoring `per_wear` on its own reference data would read ~0 by construction and prove
nothing.

| mode | range (px) | spread | vs `none` |
|---|---|---|---|
| `none` | 0.93 – 5.09 | 4.17 | — |
| `stale` | 1.08 – 4.49 | 3.41 | −18 % |
| **`per_wear`** | **0.78 – 2.58** | **1.80** | **−57 %** |

Calibration window: 71–213 frames per session. All three modes produce identical row counts and
identical trial identity — asserted, not assumed.

**The ordering is the SO4 result in miniature.** Calibrating each wear collapses the spread;
reusing the previous wear's calibration recovers only a fraction of that. Two details worth
carrying into Discussion:

- **A stale calibration can be worse than none at all.** Session 7 goes 2.97 → 4.49 px under
  `stale`, session 9 goes 2.13 → 4.42 px. Applying the wrong mount's offset actively injects
  error.
- **Session 6 is the one mount `per_wear` degrades** (1.72 → 2.48 px). Its reference window
  coincides with a documented session-start artifact. Explained, not mysterious — see the SO4 doc
  §4b.

These are resting-displacement numbers, **not force error**. The force number comes from
notebook 03+. Do not let the two merge.

In [ ]:
CALIBRATION_MODES = ('per_wear', 'none', 'stale')
CALIBRATED = ('dx', 'dy', 'dA')     # mag is rebuilt from dx/dy, never offset directly


def rest_offset(df):
    """Per-marker resting level from the start of the session, before anything happened.

    Trial 0's pre-cue interval minus the braced second. Requires add_flags to have
    run -- reusing the flags is what keeps this window and Section 4's definition of
    "bracing" from drifting apart.
    """
    window = (df['trial_id'] == 0) & (df['phase'] == 'pre_cue') & (~df['braced'])

    return {
        c: df.loc[window, [f'{c}_{i}' for i in range(N_MARKERS)]].median().to_numpy()
        for c in CALIBRATED
    }


def apply_calibration(df, offset):
    """Subtract a resting offset and rebuild mag. Runs BEFORE add_channels.

    offset=None is mode 'none' -- the frame is returned untouched, so the three modes
    differ only in the numbers, never in which rows or trials survive.
    """
    if offset is None:
        return df

    vals = {
        c: df[[f'{c}_{i}' for i in range(N_MARKERS)]].to_numpy() - offset[c]
        for c in CALIBRATED
    }

    # mag is a stored column, not derived at read time. Left alone it would keep
    # describing the uncalibrated displacement, and max_disp would inherit that.
    vals['mag'] = np.hypot(vals['dx'], vals['dy'])

    out = df.copy()
    for c, v in vals.items():
        out[[f'{c}_{i}' for i in range(N_MARKERS)]] = v     # one bulk write per channel

    return out

In [ ]:
# Filtered + flagged frames, held once. The three modes then branch off identical rows,
# which is what makes "fold membership cannot depend on the mode" structural rather than
# something to remember. Layout is reference_layout -- Section 1 asserted all 9 match.
BASE = {}
for session_dir in SESSIONS:
    n = session_number(session_dir)
    df, _ = load_session(session_dir)
    BASE[n] = add_flags(drop_uncued(session_dir, df))

OFFSETS = {n: rest_offset(df) for n, df in BASE.items()}

# Previous session, wrapping: index -1 is the last element, so session 1 takes session 10.
_ordered = sorted(OFFSETS)
STALE_DONOR = {n: _ordered[i - 1] for i, n in enumerate(_ordered)}


def offset_for(mode, n):
    if mode == 'none':
        return None
    if mode == 'per_wear':
        return OFFSETS[n]
    if mode == 'stale':
        return OFFSETS[STALE_DONOR[n]]
    raise ValueError(f'unknown calibration mode: {mode!r}')


window_frames = {
    n: int(((df['trial_id'] == 0) & (df['phase'] == 'pre_cue') & (~df['braced'])).sum())
    for n, df in BASE.items()
}

# A thin calibration window makes the offset a median of noise, and a bad reference is
# the single largest failure mode of this design (a contaminated one cost +326%).
assert min(window_frames.values()) >= 50, window_frames

for n, off in OFFSETS.items():
    for c in CALIBRATED:
        assert off[c].shape == (N_MARKERS,), (n, c, off[c].shape)
        assert not np.isnan(off[c]).any(), (n, c)

print(f'calibration window frames: {window_frames}')
print(f'stale donors: {STALE_DONOR}')

In [ ]:
def resting_level(df):
    """Median resting displacement, scored on post-cue rest frames.

    Deliberately DISJOINT from the calibration window (trial 0, pre-cue): scoring
    per_wear on its own reference data would return ~0 by construction and prove
    nothing.
    """
    m = (df['target_pct'] == 0) & (df['phase'] == 'post_cue')
    mag = df.loc[m, [f'mag_{i}' for i in range(N_MARKERS)]].to_numpy()

    return float(np.median(np.median(mag, axis=1)))


levels = {}
identity = {}

for mode in CALIBRATION_MODES:
    calibrated = {n: apply_calibration(df, offset_for(mode, n)) for n, df in BASE.items()}

    levels[mode] = {n: resting_level(df) for n, df in calibrated.items()}
    identity[mode] = {n: df['trial_id'].to_numpy() for n, df in calibrated.items()}

spread = {m: max(v.values()) - min(v.values()) for m, v in levels.items()}

# The contrast is only meaningful if the three modes differ in VALUES alone. If any of
# them moved a row or a trial, the folds would differ and the comparison would be
# between two different experiments.
for mode in CALIBRATION_MODES:
    for n in BASE:
        assert np.array_equal(identity[mode][n], identity['none'][n]), (mode, n)

# Proof the calibration did something, before any model is trained. If this fails, the
# three-mode contrast has nothing to measure.
assert spread['per_wear'] < spread['none'], spread

for mode in CALIBRATION_MODES:
    v = levels[mode]
    print(f"{mode:<9} {min(v.values()):.2f}-{max(v.values()):.2f} px  "
          f"(spread {spread[mode]:.2f})  " + ' '.join(f'{n}:{x:.2f}' for n, x in v.items()))

## Section 7 — Write the per-session cache

One `features_table.csv` + `features_meta.json` per session folder. **No merged file** — Section 8
concatenates in RAM, because a merged file is two derivation steps away from the record and that
is where staleness bites hardest.

The sidecar records the calibration mode and the git commit, so a cache can never be silently
mistaken for one built under different settings.

In [ ]:
import subprocess
from datetime import datetime

GIT_COMMIT = subprocess.run(
    ['git', 'rev-parse', '--short', 'HEAD'],
    cwd=ROOT_FOLDER, capture_output=True, text=True,
).stdout.strip()

PER_MARKER = list(CALIBRATED) + ['mag'] + list(DERIVED_GRIDS)


def build_session(n, mode):
    """load -> filter -> flag -> calibrate -> derive, for one session.

    Calibration runs before add_channels so strain, velocity and max_disp all
    inherit the correction from dx/dy rather than each needing their own.
    """
    df = apply_calibration(BASE[n], offset_for(mode, n))

    return add_channels(df, reference_layout)


def write_session(session_dir, df, n, mode):
    df.to_csv(session_dir / 'features_table.csv', index=False)

    meta = {
        'schema_version': 1,
        'built_at': datetime.now().isoformat(timespec='seconds'),
        'git_commit': GIT_COMMIT,
        'source': 'training_table.csv',
        'calibration_mode': mode,
        'stale_donor': STALE_DONOR[n] if mode == 'stale' else None,
        'per_marker_channels': PER_MARKER,
        'scalar_channels': list(DERIVED_SCALARS),
        'n_rows': len(df),
        'n_columns': df.shape[1],
    }

    with (session_dir / 'features_meta.json').open('w') as f:
        json.dump(meta, f, indent=2)

In [ ]:
SESSION_FRAMES = {}

for session_dir in SESSIONS:
    n = session_number(session_dir)
    SESSION_FRAMES[n] = build_session(n, CALIBRATION_MODE)

    if REBUILD_FEATURES:
        write_session(session_dir, SESSION_FRAMES[n], n, CALIBRATION_MODE)

assert sum(len(df) for df in SESSION_FRAMES.values()) == TOTAL_ROWS_FILTERED

# The cache on disk must match what is in RAM. Reading the mode back rather than
# trusting the variable is what catches a stale cache from an earlier run.
for session_dir in SESSIONS:
    n = session_number(session_dir)
    with (session_dir / 'features_meta.json').open() as f:
        meta = json.load(f)

    assert meta['calibration_mode'] == CALIBRATION_MODE, (n, meta['calibration_mode'])
    assert meta['n_rows'] == len(SESSION_FRAMES[n]), n
    assert meta['n_columns'] == N_COLUMNS_DERIVED, n

print(f"{len(SESSION_FRAMES)} sessions cached, mode='{CALIBRATION_MODE}', commit {GIT_COMMIT}")
print(f'rows {TOTAL_ROWS_FILTERED:,}  columns {N_COLUMNS_DERIVED}')

## Section 8 — Concatenate

`participant_id` and `session_id` are added **before** stacking. Once rows are merged, which
mount a row came from is unrecoverable — and that column is the whole cross-mount evaluation.
`participant_id` is constant today; it exists so nothing needs rewriting when participant 002
arrives.

In [ ]:
with (PARTICIPANT / 'participant.json').open() as f:
    PARTICIPANT_ID = json.load(f)['participant_id']       # the de-identified code, not a name

ID_COLUMNS = ['participant_id', 'session_id', 'trial_id']

DATASET = pd.concat(
    [
        df.assign(participant_id=PARTICIPANT_ID, session_id=n)
        for n, df in SESSION_FRAMES.items()
    ],
    ignore_index=True,
)

# Identity first, so a glance at the frame answers "which mount is this row from".
DATASET = DATASET[ID_COLUMNS + [c for c in DATASET.columns if c not in ID_COLUMNS]]

assert len(DATASET) == TOTAL_ROWS_FILTERED, len(DATASET)
assert DATASET.shape[1] == N_COLUMNS_DERIVED + 2, DATASET.shape

print(f'{len(DATASET):,} rows x {DATASET.shape[1]} columns')
print(f'participant {PARTICIPANT_ID}, sessions '
      f'{sorted(int(n) for n in DATASET.session_id.unique())}')

## Section 9 — Folds

Three evaluation axes, built from **identity only** — never from feature values, so the
calibration mode cannot move a fold.

| | Held out | Answers | |
|---|---|---|---|
| **9 — cross-mount** | a whole session | Does it survive remounting? | **The SO4 headline** |
| **9b — within-mount** | 20 % of trials, pooled | Can it read grip force at all? | Secondary |
| **9c — cross-subject** | a whole participant | Does it survive a new user? | Scaffold; needs p002 |

9b is the contrast that makes 9 interpretable: only one thing differs between them — whether the
test mount was in training — so the gap is the cost of remounting.

In [ ]:
# 9 -- cross-mount. One fold per mount, deterministic: no seed, no shuffle.
#
# int() is not cosmetic: pandas hands back np.int64, which json.dump refuses, so the
# manifest in section 10 would die on an id that prints identically to a plain int.
ALL_SESSIONS = sorted(int(n) for n in DATASET['session_id'].unique())

SESSION_FOLDS = [
    {
        'fold': i,
        'test_session': n,
        'train_sessions': [m for m in ALL_SESSIONS if m != n],
    }
    for i, n in enumerate(ALL_SESSIONS)
]


# 9b -- within-mount. A trial is (session_id, trial_id): trial_id alone repeats across
# sessions, so splitting on it would scatter nine different trials into one bucket.
TEST_FRACTION = 0.2

ALL_TRIALS = sorted(
    {(int(s), int(t)) for s, t in zip(DATASET['session_id'], DATASET['trial_id'])}
)


def trial_split(trials, seed):
    rng = np.random.default_rng(seed)
    order = rng.permutation(len(trials))
    n_test = round(len(trials) * TEST_FRACTION)

    test = {trials[i] for i in order[:n_test]}

    return sorted(set(trials) - test), sorted(test)


TRAIN_TRIALS, TEST_TRIALS = trial_split(ALL_TRIALS, RANDOM_SEED)


# 9c -- cross-subject. Built now, nothing to run it on.
def subject_folds(df):
    groups = sorted(df['participant_id'].unique())

    if len(groups) < 2:
        raise NotImplementedError(
            f'leave-one-subject-out needs 2+ participants, found {len(groups)}: {groups}. '
            f'Collect participant002 first -- see docs/design/methods/so4-calibration-protocol.md.'
        )

    return [
        {'fold': i, 'test_participant': p, 'train_participants': [q for q in groups if q != p]}
        for i, p in enumerate(groups)
    ]


# Fails loudly rather than silently producing one useless fold.
try:
    subject_folds(DATASET)
    raise AssertionError('expected NotImplementedError -- does participant002 exist now?')
except NotImplementedError as exc:
    print(f'9c scaffold: {exc}')

print(f'9  cross-mount:  {len(SESSION_FOLDS)} folds, one per session {ALL_SESSIONS}')
print(f'9b within-mount: {len(TRAIN_TRIALS)} train / {len(TEST_TRIALS)} test trials '
      f'(seed {RANDOM_SEED})')

## Section 10 — Write `split_manifest.json`

Goes in `notebooks/`, not `recordings/` — `recordings/` is gitignored, and the point of this file
is that "which mounts and trials were held out in fold 5?" stays answerable **from the committed
repo** after `nbstripout` erases the outputs.

In [ ]:
SPLIT_MANIFEST = CURRENT_FOLDER / 'split_manifest.json'

manifest = {
    'schema_version': 1,
    'built_at': datetime.now().isoformat(timespec='seconds'),
    'git_commit': GIT_COMMIT,
    'random_seed': RANDOM_SEED,
    'calibration_mode': CALIBRATION_MODE,
    'participants': [PARTICIPANT_ID],
    'n_rows': len(DATASET),
    'n_trials': len(ALL_TRIALS),
    'session_folds': SESSION_FOLDS,
    'trial_split': {
        'test_fraction': TEST_FRACTION,
        'stratified': False,      # the protocol is balanced by construction: 5 levels x 6 reps
        # (session_id, trial_id) pairs -- JSON has no tuples, so these land as [s, t] lists.
        'train': [list(t) for t in TRAIN_TRIALS],
        'test': [list(t) for t in TEST_TRIALS],
    },
    'subject_folds': None,        # needs participant002; see subject_folds() in section 9
}

with SPLIT_MANIFEST.open('w') as f:
    json.dump(manifest, f, indent=2)

print(f'wrote {SPLIT_MANIFEST.name}  ({SPLIT_MANIFEST.stat().st_size / 1024:.1f} KB)')

## Section 11 — Assertions

The gate. Each one catches a bug class that otherwise produces a plausible-looking wrong number.

The level-coverage check is what **replaces stratification**: it verifies the balance the block
design is supposed to guarantee (5 levels × 6 reps), rather than imposing it at split time.

In [ ]:
LEVELS = {0, 5, 20, 50, 80}

trial_level = DATASET.groupby(ID_COLUMNS)['target_pct'].first()

# --- the merged table -------------------------------------------------------------
assert len(DATASET) == TOTAL_ROWS_FILTERED
assert DATASET['session_id'].notna().all() and DATASET['participant_id'].notna().all()
assert set(DATASET['target_pct'].unique()) == LEVELS, sorted(DATASET['target_pct'].unique())

# --- cross-mount folds ------------------------------------------------------------
assert len(SESSION_FOLDS) == 9, len(SESSION_FOLDS)

for fold in SESSION_FOLDS:
    # The leakage check. A mount on both sides makes the headline number meaningless.
    assert fold['test_session'] not in fold['train_sessions'], fold
    assert len(fold['train_sessions']) == 8, fold

    held = DATASET[DATASET['session_id'] == fold['test_session']]

    # Replaces stratification -- verifies the block design's balance rather than forcing it.
    assert set(held['target_pct'].unique()) == LEVELS, (fold['fold'], sorted(set(held['target_pct'])))

    n_trials = held['trial_id'].nunique()
    expected = 29 if fold['test_session'] == 2 else 30      # session 2 lost trial 13
    assert n_trials == expected, (fold['test_session'], n_trials, expected)

# --- within-mount split -----------------------------------------------------------
assert set(TRAIN_TRIALS) & set(TEST_TRIALS) == set()
assert len(TRAIN_TRIALS) + len(TEST_TRIALS) == len(ALL_TRIALS) == TOTAL_TRIALS

for name, side in (('train', TRAIN_TRIALS), ('test', TEST_TRIALS)):
    levels = {int(trial_level[(PARTICIPANT_ID, s, t)]) for s, t in side}
    assert levels == LEVELS, (name, sorted(levels))

# Seeded means reproducible, or the manifest describes a split that no longer exists.
assert trial_split(ALL_TRIALS, RANDOM_SEED) == (TRAIN_TRIALS, TEST_TRIALS)

# --- folds must not depend on the calibration mode --------------------------------
# BASE is pre-calibration. If the trial universe matches, fold inputs are identity-only.
base_trials = sorted({(n, int(t)) for n, df in BASE.items() for t in df['trial_id'].unique()})
assert base_trials == ALL_TRIALS

print('all assertions passed')
print(f'  {len(DATASET):,} rows, {TOTAL_TRIALS} trials, {len(SESSION_FOLDS)} cross-mount folds')
print(f'  within-mount {len(TRAIN_TRIALS)}/{len(TEST_TRIALS)}, all 5 levels both sides')

## Section 12 — Summary

`nbstripout` erases outputs on commit, so every number worth keeping lives here.

### What was dropped, and it is only this

| | On disk | Used | Why |
|---|---|---|---|
| Sessions | 10 | **9** | Session 4's BIOPAC export was corrupted — it has `markers.csv` but no `training_table.csv`. Unlabelled, not broken |
| Trials | 270 | **269** | Session 2 trial 13 had no `grip_start` event. The operator double-pressed; it was never cued, so it is a non-trial rather than a hard trial |
| Frames | 28,261 | **28,260** | That trial was 1 frame long |

Everything else is **flagged, not deleted** — including the 8 carryover trials and all 13,705
pre-cue frames. Deleting the hard frames would shrink the error bars for free, which is the exact
criticism this thesis makes of discrete-window designs.

### The dataset

| | |
|---|---|
| Rows | **28,260** |
| Columns | **808** — 405 from `training_table.csv` + `to_cue` + 3 flags + 397 derived + 2 identity |
| Trials | **269** |
| Target | `pct_mvc_corr`, continuous |
| Calibration | `per_wear` |

### Per session

| Session | 1 | 2 | 3 | 5 | 6 | 7 | 8 | 9 | 10 |
|---|---|---|---|---|---|---|---|---|---|
| Rows | 3,667 | 3,623 | 3,384 | 2,987 | 2,771 | 2,751 | 2,927 | 3,080 | 3,070 |
| Trials | 30 | **29** | 30 | 30 | 30 | 30 | 30 | 30 | 30 |
| 9b test trials | 5 | 5 | 7 | 8 | 6 | 4 | 6 | 6 | 7 |

Trials per level: **54** each at 0 / 5 / 20 / 50 % MVC, **53** at 80 % — session 2's lost trial.
Row counts differ by session, so never reuse a mask or index array across them.

### The folds

| | Held out | Count | |
|---|---|---|---|
| **Cross-mount** | one whole session | **9 folds** | The SO4 headline. 30 held-out trials each, 29 for session 2 |
| **Within-mount** | 20 % of trials, pooled | 215 train / 54 test | Seeded (42). The contrast that makes the headline interpretable |
| **Cross-subject** | one whole participant | — | Scaffold only. Raises `NotImplementedError` until participant 002 exists |

Folds are built from **identity only**, never from feature values, so the calibration mode cannot
move them. Not stratified — the protocol is balanced by construction (5 levels × 6 reps), and the
assertions verify that balance rather than imposing it.

### Flags

| Flag | Frames | |
|---|---|---|
| `pre_cue` | 13,705 | 48 % of the dataset. At ~0.7–1.1 % MVC whatever the trial is labelled |
| `post_cue` | 14,555 | |
| `braced` | 4,365 | Final 1 s before the cue. Subset of `pre_cue` |
| `carryover` | 959 | 8 trials: s1 t0/t2/t5/t13, s2 t10, s5 t12/t13, s8 t5. **Four of eight are session 1** |

`vx`/`vy` are `NaN` on **269 frames** (0.95 %) — frame 0 and every trial boundary, where there is
no earlier frame to difference against. Left bare; notebook 03 decides how to handle them.

### Calibration

Resting spread across the 9 mounts, scored on rest frames outside the calibration window:

| mode | spread (px) | vs `none` |
|---|---|---|
| `none` | 4.17 | — |
| `stale` | 3.41 | −18 % |
| **`per_wear`** | **1.80** | **−57 %** |

A stale calibration was **worse than none** on two mounts. Full discussion:
`docs/design/methods/so4-calibration-protocol.md`.

### Artifacts written

| Path | |
|---|---|
| `recordings/…/sessionN_…/features_table.csv` | 9 caches, 45 MB each. Gitignored, regenerable in ~80 s |
| `recordings/…/sessionN_…/features_meta.json` | Records calibration mode + git commit |
| `notebooks/split_manifest.json` | **Committed.** The only record of fold membership after outputs are stripped |

### What notebook 03 gets

`DATASET` (28,260 × 808), `SESSION_FOLDS` (9), `TRAIN_TRIALS` / `TEST_TRIALS`, and
`subject_folds()` waiting on a second participant.

Open before notebook 03 starts: nested CV or one fixed inner split · does the baseline get tuned
at all · ablation of the surviving channels, on the inner fold only.